# Phase 12 — DistilRoBERTa Fine-tuning (train+valid)

Fine-tuning του `distilroberta-base` σε train+valid για Food Hazard Detection.

**Γιατί DistilRoBERTa:**
- Διαφορετική αρχιτεκτονική από DistilBERT → κάνει διαφορετικά λάθη
- Εκπαιδευμένο με RoBERTa objectives (no NSP, dynamic masking)
- Καλός υποψήφιος για ensemble με DistilBERT

**Στρατηγική:** train+valid (5647 δείγματα), 20 epochs, χωρίς early stopping

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Συνδυασμός train + valid
train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

print(f'Train+Valid: {len(texts_full)}')
print(f'Test:        {len(texts_test)}')

In [ ]:
le_hazard  = LabelEncoder()
le_product = LabelEncoder()

le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_full_product = le_product.transform(train_full['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Hazard classes:  {n_hazard}')
print(f'Product classes: {n_product}')

In [ ]:
MODEL_NAME   = 'distilroberta-base'
BATCH_SIZE   = 32
MAX_LENGTH   = 128
LR           = 2e-5
N_EPOCHS     = 20
WARMUP_RATIO = 0.1

print(f'Φόρτωση tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded!')

dummy_labels = np.zeros(len(texts_test), dtype=int)

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)


def get_predictions(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = model(**batch).logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

In [ ]:
full_loader_hazard  = DataLoader(FoodHazardDataset(texts_full, y_full_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
full_loader_product = DataLoader(FoodHazardDataset(texts_full, y_full_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
test_loader_hazard  = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_product = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(full_loader_hazard)}')
print(f'Batch size: {BATCH_SIZE} | Epochs: {N_EPOCHS} | LR: {LR}')

## Εκπαίδευση DistilRoBERTa — Hazard Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ DistilRoBERTa (train+valid) — HAZARD ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS}\n')

roberta_hazard = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_hazard
).to(device)

optimizer_hazard = AdamW(roberta_hazard.parameters(), lr=LR, weight_decay=0.01)
total_steps      = len(full_loader_hazard) * N_EPOCHS
warmup_steps     = int(total_steps * WARMUP_RATIO)
scheduler_hazard = get_linear_schedule_with_warmup(
    optimizer_hazard,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(roberta_hazard, full_loader_hazard, optimizer_hazard, scheduler_hazard)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

print('\nΕκπαίδευση Hazard ολοκληρώθηκε!')

## Εκπαίδευση DistilRoBERTa — Product Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ DistilRoBERTa (train+valid) — PRODUCT ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS}\n')

roberta_product = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_product
).to(device)

optimizer_product = AdamW(roberta_product.parameters(), lr=LR, weight_decay=0.01)
total_steps       = len(full_loader_product) * N_EPOCHS
warmup_steps      = int(total_steps * WARMUP_RATIO)
scheduler_product = get_linear_schedule_with_warmup(
    optimizer_product,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(roberta_product, full_loader_product, optimizer_product, scheduler_product)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

print('\nΕκπαίδευση Product ολοκληρώθηκε!')

## Αποθήκευση Probabilities + Submission

In [ ]:
# Test probabilities για ensemble
roberta_test_hazard_probs  = get_probabilities(roberta_hazard,  test_loader_hazard)
roberta_test_product_probs = get_probabilities(roberta_product, test_loader_product)

np.save('roberta_test_hazard_probs.npy',  roberta_test_hazard_probs)
np.save('roberta_test_product_probs.npy', roberta_test_product_probs)

print(f'Test hazard probs:  {roberta_test_hazard_probs.shape}')
print(f'Test product probs: {roberta_test_product_probs.shape}')
print('Αποθηκεύτηκαν τα .npy για ensemble')

# Submission DistilRoBERTa μόνο
test_hazard  = le_hazard.inverse_transform(roberta_test_hazard_probs.argmax(axis=1))
test_product = le_product.inverse_transform(roberta_test_product_probs.argmax(axis=1))

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_distilroberta.csv', index=False)

print('Αποθηκεύτηκε: submission_distilroberta.csv')
